# Running the experiments on Google Colab

Runs the code of this repository on a Colab GPU (we used an A100). A Colab runtime starts empty, so cells 2-3 run once per runtime (about 5 min).

On your own machine, make the code bundle with `python src/colab_pack.py` (it writes `freqcfg.zip`). In Colab choose Runtime > Change runtime type > GPU,
drag `freqcfg.zip` into the Files panel on the left, then run the cells in order.

Stop the runtime when you are done (Runtime > Disconnect and delete runtime): an idle runtime still uses compute units.
Colab also stops a runtime that has no connected browser for a while, and everything on it is lost, so save results often (cell 7).

In [ ]:
# @title 1. Which GPU did we get?
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# @title 2. Get the code into /content/freqcfg
import os, zipfile
os.chdir('/content')
if not os.path.exists('/content/freqcfg/src/common.py'):
    zips = [z for z in ['/content/freqcfg.zip', '/content/drive/MyDrive/freqcfg.zip'] if os.path.exists(z)]
    if not zips:
        raise SystemExit('No code found. Drag freqcfg.zip into the Files panel on the left (it lands in /content), then run this cell again.')
    zipfile.ZipFile(zips[0]).extractall('/content')
print(sorted(f for f in os.listdir('/content/freqcfg/src') if f.endswith('.py'))[:8], '...')

In [ ]:
# @title 3. Setup: packages, EDM code, 4 checkpoints, CIFAR-10, classifiers (about 3-5 min, skips what is done)
!bash /content/freqcfg/scripts/setup.sh

In [ ]:
# @title 4. Speed test: 1000 images with plain CFG (w=3), against the A10 we used first
import os, sys, time, torch
os.chdir('/content/freqcfg/src'); sys.path.insert(0, '/content/freqcfg/src')
from common import load_nets, sample, ConstCFG
dev = 'cuda'; net_c, net_u = load_nets(dev)
labels = torch.arange(500, device=dev) % 10
sample(net_c, net_u, labels[:100], ConstCFG(3.0), seed=0, device=dev)          # warm-up
torch.cuda.synchronize(); t = time.time()
for s in (0, 500): sample(net_c, net_u, labels, ConstCFG(3.0), seed=s, device=dev)
torch.cuda.synchronize(); per = (time.time() - t) / 1000
A10 = 0.090                                                                         # s per image on the A10 (5k arms: about 449 s)
print(f'{torch.cuda.get_device_name()}: {per * 1000:.1f} ms per image, {A10 / per:.2f}x the A10 speed; a 5k arm takes about {5000 * per / 60 + 2:.1f} min with metrics')

In [ ]:
# @title 5. Check: redo "no push" and "correct answer" for "not a cat" (k=3) and compare with the A10 run
# A10, same seeds: no push FID 7.61, IS 9.12, cats 9.0%; correct answer FID 6.42, IS 9.21, cats 1.5%.
# Small differences are expected (another GPU type gives slightly different fp16 sums; on the A100 we got FID 7.72 and 6.41).
!cd /content/freqcfg/src && python neg.py base --k 3

In [ ]:
# @title 6. Run experiments (edit the lines below)
# !cd /content/freqcfg/src && python neg.py arm --k 3 --a 0 --b 1 --lo 0.8
# !bash /content/freqcfg/scripts/sweep10.sh > /content/freqcfg/src/sweep10.log 2>&1

In [ ]:
# @title 7. Save results: /content/results.zip (no 5k sample files; add them with KEEP_SAMPLES = True)
import os, glob, zipfile
KEEP_SAMPLES = False
os.chdir('/content/freqcfg/src')
files = [f for f in glob.glob('out/**/*', recursive=True) if os.path.isfile(f) and not f.startswith(('out/data', 'out/fid_cache'))
         and (KEEP_SAMPLES or not f.endswith('.npz') or 'spectrum' in f)] + glob.glob('*.log')
with zipfile.ZipFile('/content/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(f)
print(len(files), 'files in /content/results.zip')
try:
    from google.colab import files as colab_files; colab_files.download('/content/results.zip')   # browser only
except Exception: pass